# CLAUDS AION ILC magnitude adapter

Train a linear adjustment matrix `M` for the magnitudes seen by frozen AION:

`m_aion_grizy = m_input_grizy + M @ standardized([g,r,i,z,y,u,u*,Y,J,Ks])`

Only `M` is trained here. AION and the AION-only photo-z head are frozen.

In [1]:
from pathlib import Path

import aion_magnitude as am
import aion_magnitude.ilc as ilc


In [2]:
CATALOGUE_PATH = Path("data/clauds/COSMOS-HSCpipe-Phosphoros.fits")
CACHE_ROOT = Path("cache")

RUN_MODES = ("grizy_only", "u_u_star_Y", "all")

BASE_CONFIG = ilc.AIONILCConfig(
    catalogue_path=CATALOGUE_PATH,
    max_rows=None,
    cache_root=CACHE_ROOT,
    output_dir=CACHE_ROOT / "aion_ilc",
    source_bands=("g", "r", "i", "z", "y", "u", "u_star", "Y", "J", "Ks"),
    source_invalid_fill="median",
    epochs=10,
    train_batch_size=128,
    eval_batch_size=256,
    learning_rate=1e-2,
    optimizer_kind="spsa",
    finite_diff_step=0.03,
    prior_kind="uniform",
    gaussian_sigma=0.3,
    prior_weight=0.0,
    uniform_bound=1.0,
    delta_clip_mag=None,
    split_strategy="random",
    train_fraction=0.20,
    test_fraction=0.75,
    val_fraction=0.05,
    device_choice="auto",
)

BASE_CONFIG


AIONILCConfig(catalogue_path=PosixPath('data/clauds/COSMOS-HSCpipe-Phosphoros.fits'), max_rows=None, cache_root=PosixPath('cache'), split_output_dir=None, output_dir=PosixPath('cache/aion_ilc'), solution_path=None, mode='u_u_star_Y', source_bands=('g', 'r', 'i', 'z', 'y', 'u', 'u_star', 'Y', 'J', 'Ks'), target_bands=['g', 'r', 'i', 'z', 'y'], source_invalid_fill='median', delta_clip_mag=None, aion_head_checkpoint_path=None, split_chunk_size=250000, overwrite_split_cache=False, field_column=None, split_strategy='random', train_fraction=0.2, test_fraction=0.75, val_fraction=0.05, test_fields=[], target_redshift_column='ZPHOT', z_min=0.0, z_max=6.0, n_z_bins=300, mag_zero_point=23.0, hsc_mag_faint_limits={'g': 24.5, 'r': 24.5, 'i': 24.0, 'z': 24.5, 'y': 24.5}, epochs=10, train_batch_size=128, eval_batch_size=256, learning_rate=0.01, optimizer_kind='spsa', finite_diff_step=0.03, spsa_beta1=0.9, spsa_beta2=0.99, spsa_eps=1e-08, prior_kind='uniform', prior_weight=0.0, gaussian_sigma=0.3, uni

In [3]:
# Check whether the frozen AION input path is differentiable before launching a long run.
gradient_checks = {
    mode: ilc.check_ilc_gradient_flow(
        BASE_CONFIG,
        mode=mode,
        train_batch_size=8,
        max_train_batches=1,
        max_val_batches=1,
    )
    for mode in RUN_MODES
}

gradient_checks


{'grizy_only': {'mode': 'grizy_only',
  'logits_requires_grad': False,
  'loss_requires_grad': False,
  'loss': 3.8448688983917236,
  'matrix_grad_is_none': True,
  'matrix_grad_abs_max': None,
  'matrix_grad_abs_sum': None,
  'backward_error': None},
 'u_u_star_Y': {'mode': 'u_u_star_Y',
  'logits_requires_grad': False,
  'loss_requires_grad': False,
  'loss': 3.8448688983917236,
  'matrix_grad_is_none': True,
  'matrix_grad_abs_max': None,
  'matrix_grad_abs_sum': None,
  'backward_error': None},
 'all': {'mode': 'all',
  'logits_requires_grad': False,
  'loss_requires_grad': False,
  'loss': 3.8448688983917236,
  'matrix_grad_is_none': True,
  'matrix_grad_abs_max': None,
  'matrix_grad_abs_sum': None,
  'backward_error': None}}

In [4]:
# Autograd is blocked, but finite M steps can still change AION's black-box output.
finite_step_checks = {
    mode: ilc.check_ilc_finite_step_sensitivity(
        BASE_CONFIG,
        mode=mode,
        pattern="identity" if mode == "grizy_only" else "active_random",
        step_values=(0.01, 0.03, 0.1, 0.3, 1.0),
        train_batch_size=32,
        max_train_batches=1,
        max_val_batches=1,
    )
    for mode in RUN_MODES
}

finite_step_checks


{'grizy_only': {'mode': 'grizy_only',
  'pattern': 'identity',
  'baseline_cross_entropy': 3.795560359954834,
  'steps': [{'step': 0.01,
    'cross_entropy': 3.803158760070801,
    'delta_cross_entropy': 0.007598400115966797,
    'mean_abs_logit_delta': 0.2100542038679123,
    'max_abs_logit_delta': 2.8501787185668945},
   {'step': 0.03,
    'cross_entropy': 3.8021187782287598,
    'delta_cross_entropy': 0.006558418273925781,
    'mean_abs_logit_delta': 0.28005823493003845,
    'max_abs_logit_delta': 3.4770259857177734},
   {'step': 0.1,
    'cross_entropy': 3.816978931427002,
    'delta_cross_entropy': 0.02141857147216797,
    'mean_abs_logit_delta': 0.4727064073085785,
    'max_abs_logit_delta': 5.132296562194824},
   {'step': 0.3,
    'cross_entropy': 3.7424426078796387,
    'delta_cross_entropy': -0.05311775207519531,
    'mean_abs_logit_delta': 0.6891843676567078,
    'max_abs_logit_delta': 4.624264717102051},
   {'step': 1.0,
    'cross_entropy': 4.191086292266846,
    'delta_cro

In [9]:
RUN_TRAINING = True

if RUN_TRAINING:
    if BASE_CONFIG.optimizer_kind == "autograd":
        broken_modes = [
            mode for mode, check in gradient_checks.items()
            if not check["loss_requires_grad"] or check["matrix_grad_is_none"]
        ]
        if broken_modes:
            raise RuntimeError(f"ILC gradient training is blocked for modes: {broken_modes}")
    ilc_results = ilc.train_aion_ilc_modes(RUN_MODES, BASE_CONFIG)
else:
    ilc_results = {}

ilc_results.keys()


ilc mode=grizy_only epoch=000 train_loss=4.0767 val_loss=4.0563 val_nmad=0.1254
ilc mode=grizy_only epoch=001 train_loss=4.1109 val_loss=4.0862 val_nmad=0.1285
ilc mode=grizy_only epoch=002 train_loss=4.0978 val_loss=4.0240 val_nmad=0.1171
ilc mode=grizy_only epoch=003 train_loss=4.1130 val_loss=4.0005 val_nmad=0.1120
ilc mode=grizy_only epoch=004 train_loss=4.0886 val_loss=3.9974 val_nmad=0.1110
ilc mode=grizy_only epoch=005 train_loss=4.0669 val_loss=3.9896 val_nmad=0.1057
ilc mode=grizy_only epoch=006 train_loss=4.0621 val_loss=3.9854 val_nmad=0.1087
ilc mode=grizy_only epoch=007 train_loss=4.0758 val_loss=3.9558 val_nmad=0.1068
ilc mode=grizy_only epoch=008 train_loss=4.0969 val_loss=4.1175 val_nmad=0.1199
ilc mode=grizy_only epoch=009 train_loss=4.0975 val_loss=4.0089 val_nmad=0.1075
ilc mode=u_u_star_Y epoch=000 train_loss=4.2571 val_loss=4.2664 val_nmad=0.1557
ilc mode=u_u_star_Y epoch=001 train_loss=4.2452 val_loss=4.1607 val_nmad=0.1368
ilc mode=u_u_star_Y epoch=002 train_loss

dict_keys(['grizy_only', 'u_u_star_Y', 'all'])

In [10]:
for mode, result in ilc_results.items():
    print("\nMODE", mode)
    print("solution:", result["solution_path"])
    print("final val:", result["final_val_metrics"])
    ilc.print_ilc_matrix(result)



MODE grizy_only
solution: cache/aion_ilc/aion_ilc_COSMOS_HSCpipe_Phosphoros_zp23p0_all_grizy_only.pt
final val: {'bias': 0.02745198830962181, 'median_bias': 0.009952983818948269, 'nmad': 0.10679485648870468, 'catastrophic_outlier_fraction': 0.2788108289241791, 'cross_entropy': 3.955792385087886, 'n_batches': 142}
target/source          g          r          i          z          y          u     u_star          Y          J         Ks
            g     0.4782    -0.5532    -0.4944    -0.5789     0.5256     0.0000     0.0000     0.0000     0.0000     0.0000
            r     0.2761    -0.3857    -0.4521    -0.3111     0.2436     0.0000     0.0000     0.0000     0.0000     0.0000
            i     0.3672    -0.7212     0.0230     0.1839    -0.4964     0.0000     0.0000     0.0000     0.0000     0.0000
            z    -0.1212     0.3802    -0.3594    -0.8118     0.1424     0.0000     0.0000     0.0000     0.0000     0.0000
            y     0.6326    -0.8123    -0.8143     0.4551    -0.

In [11]:
# Pick one learned matrix and inspect the cache paths that would be used when applying it.
SELECTED_MODE = "u_u_star_Y"
SELECTED_SOLUTION_PATH = (
    ilc_results[SELECTED_MODE]["solution_path"]
    if ilc_results
    else CACHE_ROOT / "aion_ilc" / "aion_ilc_COSMOS_HSCpipe_Phosphoros_zp23p0_all_u_u_star_Y.pt"
)

adapted_config = am.AIONMagnitudeConfig(
    catalogue_path=CATALOGUE_PATH,
    max_rows=None,
    cache_root=CACHE_ROOT,
    extra_bands=(),
    use_aion_embedding=True,
    use_mlp_features=False,
    include_grizy_in_mlp=False,
    model_kinds=("aion",),
    baseline_epochs=10,
    aion_mag_adjustment_path=SELECTED_SOLUTION_PATH,
    force_recompute_embeddings=True,
    device_choice="auto",
)

am.resolve_training_paths(adapted_config)


{'run_tag': 'COSMOS_HSCpipe_Phosphoros_zp23p0_all',
 'experiment_tag': 'COSMOS_HSCpipe_Phosphoros_zp23p0_all_aiononly_ilc_aion_ilc_COSMOS_HSCpipe_Phosphoros_zp23p0_all_u_u_star_Y',
 'split_output_dir': PosixPath('cache/clauds_split_COSMOS_HSCpipe_Phosphoros_zp23p0_all'),
 'cache_path': PosixPath('cache/clauds_aion_embeddings_COSMOS_HSCpipe_Phosphoros_zp23p0_all_ilc_aion_ilc_COSMOS_HSCpipe_Phosphoros_zp23p0_all_u_u_star_Y.pt'),
 'baseline_output_dir': PosixPath('cache/baselines_COSMOS_HSCpipe_Phosphoros_zp23p0_all_aiononly_ilc_aion_ilc_COSMOS_HSCpipe_Phosphoros_zp23p0_all_u_u_star_Y')}

In [12]:
RUN_ADAPTED_AION_BASELINE = False

if RUN_ADAPTED_AION_BASELINE:
    adapted_run = am.run_training_and_evaluation(
        adapted_config,
        model_kind="aion",
        split="test",
    )
    adapted_run["baseline_results"]["aion"]["final_metrics"]["test"]
